# 07 — Hierarchical Risk Parity (HRP)

Notebooks 05 and 06 asked the same question of two different risk-based
objectives: the Most Diversified Portfolio (MDP, notebook 05) *kept* the
UUP corner that survives covariance shrinkage, and Equal Risk Contribution
(ERC, notebook 06) *shrank* it in relative terms without removing it. Both use the full Σ — cross-covariances included — in the condition that defines their weights (MDP's first-order condition, ERC's equal-risk-contribution equations).

This notebook asks the same question of a hierarchy-based method that never
inverts &Sigma; at all. Hierarchical Risk Parity (HRP) clusters assets by
correlation distance and allocates capital by walking down the resulting
tree, splitting mass at each node in inverse proportion to the variance of
the two sides' inverse-variance sub-portfolios. What happens to the UUP
corner under a method that only ever compares diagonal blocks of &Sigma;,
and does UUP sit in a cluster of its own?

## §1 Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt
import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import dendrogram, is_valid_linkage, is_monotonic, cophenet
from scipy.spatial.distance import pdist, squareform

import maplab as ml
import _nb07_helpers as h
from maplab import (
    Panel, apply_style, GMV, MaxSharpe, EqualWeight, MostDiversified,
    EqualRiskContribution, HierarchicalRiskParity, risk_contributions,
    backtest, summary, mdp_closed_form,
)
from maplab.models import (
    _min_variance_long_only, _mdp_long_only, _erc_long_only, _tangency_long_only,
    _hrp_long_only,
)

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

### §1.2 Recipe and identities

**Recipe.** C = D&#8315;&#185;&Sigma;D&#8315;&#185; (the correlation matrix);
d<sub>ij</sub> = &#8730;((1&#8722;&rho;<sub>ij</sub>)/2); cluster on the
Euclidean distance between COLUMNS of d (distance of distances, the
literature's recipe); single linkage; then recursive bisection. At every
split the two sides get capital in inverse proportion to the variance of
each side's inverse-variance sub-portfolio:
&alpha;<sub>L</sub> = V<sub>R</sub>/(V<sub>L</sub>+V<sub>R</sub>), V = w'&Sigma;w
for w = the IVP weights of that side alone. **Registered bisection is
tree-following** (split at each dendrogram node, top to bottom). The
literature's original algorithm instead splits the LEAF ORDER into equal
halves at each step ("positional"); a read-only check run before this
notebook found that positional HRP's weights change with the input column
order on 182/208 rebalances (&gt; 1% half-L1 on 116 of them), which is why
tree-following is registered here and positional is carried only as a
sensitivity.

Only the diagonal blocks &Sigma;<sub>LL</sub>, &Sigma;<sub>RR</sub> ever
enter an allocation decision; the cross block &Sigma;<sub>LR</sub> never
does — it only shapes which assets end up together in the tree.

**Identities.**
- C = I (Sigma diagonal) &#8658; HRP = inverse variance = long-only GMV,
  for any tree and either bisection. Contrast: on the same diagonal
  &Sigma;, ERC and MDP both reduce to inverse VOL, not inverse variance —
  HRP is a variance-based method, ERC/MDP are vol-based.
- N = 2 &#8658; inverse variance for any &rho;.
- Exchangeable &Sigma; (equal vols, constant &rho;) &#8658; GMV = MDP = ERC
  = EW, but positional HRP &#8800; EW for &rho; &gt; 0 whenever N is not a
  power of 2 — the halves split is size-dependent even when every asset is
  otherwise identical.
- Uniform scaling c·Σ leaves HRP's weights unchanged. Per-asset scaling KΣK leaves the tree unchanged (correlations are unchanged) but not the weights in any simple way: HRP has no analogue of ERC/MDP's rule w(KΣK) ∝ K⁻¹w(Σ).
- **Root-singleton identity.** If an asset sits alone on one side of the FINAL merge, its weight is exactly V<sub>rest</sub>/(σ<sup>2</sup> &#43; V<sub>rest</sub>), independent of its correlations with the rest of the universe — HRP is "correlation-blind" for that asset in exactly this sense.

In [ ]:
ml.inference.render_registration(ml.inference.load_registration("nb07"))

## §3 Snapshot at three dates (sample Σ)

Recomputed at 2014-12-31 / 2017-12-31 / 2022-12-31 — the same three dates
that fixed the H1a/H2b/H3 directions in the read-only design check. §3.1:
what do the weights actually look like?

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

snap = h.snapshot(panel, DATES)

print("snapshot computed at:", [d.date() for d in DATES])

In [ ]:
F4 = lambda v: f"{v:.4f}"
E3 = lambda v: f"{v:.3e}"

for asof in DATES:
    s = snap[asof]
    print(f"\n----- {asof.date()} weights -----")
    tbl = pd.DataFrame(s["W"], index=s["cols"])
    print(tbl.to_string(float_format=F4))

§3.2 — diversification: does HRP raise effN and lower the UUP weight
relative to GMV / MDP / ERC, and where does UUP sit in the correlation /
vol structure at each date?

In [ ]:
for asof in DATES:
    s = snap[asof]
    Sigma_np, iU, n = s["Sigma_np"], s["iU"], s["n"]
    C = s["C"]
    rho_bar = float((C.sum() - n) / (n * (n - 1)))
    uup_mean_corr = float((C[iU].sum() - 1.0) / (n - 1))
    uup_vol_rank = int((s["sigma_np"] <= s["sigma_np"][iU]).sum())
    print(f"\n----- {asof.date()} -----")
    print(f"UUP mean pairwise corr={F4(uup_mean_corr)}  vol rank (1=lowest)={uup_vol_rank}  rho_bar={F4(rho_bar)}")

    div_tbl = ml.diagnostics.diversification_table(s["W"], Sigma_np, tickers=s["cols"])
    print(div_tbl.to_string(float_format=F4))

§3.3 — tree facts: is UUP alone on one side of the final merge, and how
much of the total split cascades onto a single asset?

In [ ]:
for asof in DATES:
    s = snap[asof]
    Sigma_np, iU, cols = s["Sigma_np"], s["iU"], s["cols"]
    Z = s["info"]["Z"]
    coph = float(cophenet(Z, s["y"])[0])
    print(f"\n----- {asof.date()} -----")
    print(f"UUP root singleton (tree): {h.root_singleton(Z, iU)}   cophenetic corr={F4(coph)}   "
          f"last merge height={F4(Z[-1, 2])}")
    tp = h.tree_path(Sigma_np, Z, iU)
    tp_display = [(a, b, round(c, 4)) for a, b, c in tp]
    print("UUP tree path (own-side n, other-side n, share to own side):", tp_display)
    prod = 1.0
    for _, _, a in tp:
        prod *= a
    print(f"  |path product - w_UUP| = {E3(abs(prod - s['W']['HRP'][iU]))}")
    order = s["info"]["order"]
    pp = h.pos_path(Sigma_np, s["info_pos"]["order"], iU)
    pp_display = [(a, b, round(c, 4)) for a, b, c in pp]
    print("UUP positional path (own-side n, other-side n, share to own side):", pp_display,
          f" isolation depth={len(pp)}")
    print("leaf order (tree):", [cols[j] for j in order])

§3.4 — counterfactual: with UUP's correlations set to 0, does its weight
move, and by how much relative to GMV / MDP / ERC?

In [ ]:
for asof in DATES:
    s = snap[asof]
    iU = s["iU"]
    Z_cf = s["info_cf"]["Z"]
    dcf = {"HRP": abs(s["W"]["HRP"][iU] - s["w_hrp_cf"][iU])}
    dcf.update({m: abs(s["comps"][m][iU] - s["comps_cf"][m][iU]) for m in s["comps"]})
    print(f"\n----- {asof.date()} -----  UUP root singleton under Sigma_cf: {h.root_singleton(Z_cf, iU)}")
    cf_tbl = pd.DataFrame({
        "w_UUP": {"HRP": s["W"]["HRP"][iU], **{m: s["comps"][m][iU] for m in s["comps"]}},
        "w_UUP_cf": {"HRP": s["w_hrp_cf"][iU], **{m: s["comps_cf"][m][iU] for m in s["comps"]}},
        "abs_delta": dcf,
    })
    print(cf_tbl.to_string(float_format=F4))

§3.5 — recipe sensitivity: how much do the weights move under a different
linkage (average / ward), a direct distance (no distance-of-distances), or
d = √(1−ρ) instead of √((1−ρ)/2) (a sanity check that should show 0
movement, since it just rescales every pairwise distance by a constant)?

In [ ]:
for asof in DATES:
    s = snap[asof]
    w_hrp = s["W"]["HRP"]
    rec = {"alt_" + k: ml.diagnostics.half_l1(v, w_hrp) for k, v in s["alts"].items()}
    rec["sqrt(1-rho)_sanity"] = ml.diagnostics.half_l1(s["w_sanity"], w_hrp)
    rec["tree_vs_positional"] = ml.diagnostics.half_l1(s["W"]["HRP[pos]"], w_hrp)
    print(f"\n----- {asof.date()} (half-L1 vs registered HRP) -----")
    print(pd.Series(rec).to_string(float_format=E3))

### Figure F1 — snapshot map at 2022-12-31

Where do HRP, GMV, MDP, ERC, MaxSharpe and EW sit in (vol, μ) space at the
train-window snapshot that fixed the H1a/H2b/H3 directions? μ here is a
252-day trailing sample mean, so the y-axis is illustrative — the honest
reading is the x-position (risk) and the printed weights, not "expected
return". MaxSharpe is computed at this single date only (snapshot reference, not backtested); IVP is backtested in §4.

In [ ]:
asof_2022 = pd.Timestamp("2022-12-31")
mu_2022, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np = Sigma_2022.to_numpy()

s2022 = snap[asof_2022]
w_hrp_2022 = s2022["W"]["HRP"]
w_gmv_2022 = s2022["W"]["GMV"]
w_mdp_2022 = s2022["W"]["MDP"]
w_erc_2022 = s2022["W"]["ERC"]
w_ivp_2022 = s2022["W"]["IVP"]
w_ew_2022 = s2022["W"]["EW"]

rf_ann_2022 = MaxSharpe()._rf_ann(panel, asof_2022)
excess_2022 = mu_2022 - rf_ann_2022
if excess_2022.max() > 0:
    w_msr_2022, _ = _tangency_long_only(excess_2022.to_numpy(), Sigma_np, "F1", asof_2022)
else:
    w_msr_2022 = _min_variance_long_only(Sigma_2022)

fig = ml.plotting.snapshot_map(
    Sigma_2022, mu_2022,
    {"HRP": w_hrp_2022, "GMV": w_gmv_2022, "MDP": w_mdp_2022, "ERC": w_erc_2022,
     "MaxSharpe": w_msr_2022, "EqualWeight": w_ew_2022, "IVP": w_ivp_2022},
    title=f"Snapshot {asof_2022.date()} -- HRP / GMV / MDP / ERC / MaxSharpe / EW / IVP",
    figsize=(10, 6), legend_outside=True,
)
plt.show()

print(f"w_UUP: HRP={w_hrp_2022[s2022['iU']]:.4f}  GMV={w_gmv_2022[s2022['iU']]:.4f}  "
      f"MDP={w_mdp_2022[s2022['iU']]:.4f}  ERC={w_erc_2022[s2022['iU']]:.4f}  "
      f"MaxSharpe={w_msr_2022[s2022['iU']]:.4f}  EW={w_ew_2022[s2022['iU']]:.4f}  "
      f"IVP={w_ivp_2022[s2022['iU']]:.4f}")

### Figure F2 — mechanism grid: dendrogram, quasi-diagonalized correlation, UUP weight bars

Three rows (the three snapshot dates) by three panels: (a) the dendrogram
with UUP's leaf label highlighted and the root split's share to UUP's side
annotated; (b) the correlation matrix reordered into the dendrogram's leaf
order (quasi-diagonalized), with UUP's row/column boxed and a line at the
root split boundary; (c) UUP's weight under IVP / HRP / HRP[pos] / ERC /
GMV / MDP / EW, with hollow bars showing the same methods' weight under
Σ_cf (UUP's correlations zeroed). A small separate panel checks the split
identity directly: mass ratio vs. V_R/(V_L+V_R) at every dendrogram node,
every snapshot date.

(a) now shows the capital cascade: the share each node passes to its left &#124; right branch, and each leaf's final HRP weight.

In [ ]:
fig, annotated_counts = h.mechanism_grid(snap, DATES, info=snap[DATES[-1]]["info"])
for asof, annotated, n_links in annotated_counts:
    print(f"{asof.date()}: annotated {annotated}/{n_links} nodes")
plt.show()

In [ ]:
fig = h.split_identity_figure(snap, DATES)
plt.show()

## §4 Backtests

Nine runs, same cost/rebalance/warm-up harness as notebooks 01–06. `HRP(S)`
is the registered run; `GMV(S)`, `EW`, `MDP(S)`, `ERC(S)` are the
reproduction-gate comparators (already backtested in earlier notebooks);
`HRP(LW)`, `HRP[pos](S)`, `HRP[ward](S)` are sensitivities; `IVP` is the
notebook-local inverse-variance reference.

In [ ]:
strategies = {
    "HRP(S)": HierarchicalRiskParity(cov_estimator=ml.sample_cov),
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "EW": EqualWeight(),
    "MDP(S)": MostDiversified(cov_estimator=ml.sample_cov),
    "ERC(S)": EqualRiskContribution(cov_estimator=ml.sample_cov),
    "HRP(LW)": HierarchicalRiskParity(cov_estimator=ml.ledoit_wolf_cov),
    "HRP[pos](S)": HierarchicalRiskParity(bisection="positional"),
    "HRP[ward](S)": HierarchicalRiskParity(linkage="ward"),
    "IVP": ml.InverseVariance(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    assert wlog.index.equals(rdates), f"{name}: wlog index != rdates"
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

print(f"{len(results)} runs completed.")

### Reproduction gate

`GMV(S)`, `EW`, `MDP(S)`, `ERC(S)` must match notebooks 01/05/06 in
full/train/test Sharpe to 2 dp: GMV 0.62/0.59/0.76, EW 0.74/0.70/0.91,
MDP(S) 0.76/0.70/0.98, ERC(S) 0.76/0.73/0.94. If this doesn't hold, the
harness or estimator wiring changed something and everything downstream is
suspect — so this stops the notebook (`assert`) rather than just printing a
warning.

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table["cost_drag"] = summary_table["ann_turnover"] * ml.COST_BPS / 1e4
print(summary_table.to_string(float_format=F4))

In [ ]:
ml.inference.check_reproduction(summary_table, {"GMV(S)": "GMV_sample", "EW": "EW", "MDP(S)": "MDP_sample", "ERC(S)": "ERC_sample"})

print("REPRODUCTION GATE: PASSED -- GMV(S), EW, MDP(S), ERC(S) match notebooks 01/05/06 to 2 dp.")

## §5 Mechanism checks, all 208 rebalances × {sample, Ledoit-Wolf} × {tree, positional}

Recomputed fresh with `_hrp_long_only` (not read off the §4 backtests), so
this is an independent check of the closed-form arithmetic itself, not a
tautology. Every tolerance below is asserted at ≤ 1e-12 and every count at
= 208 — HRP has no solver and no iteration, so unlike ERC's coordinate
descent there is no reason to allow any slack at all.

In [ ]:
H, pos_spreads_208, cophs, root_sides = h.mechanism_checks(panel, rdates)
print(H.to_string(float_format=E3))

tol = 1e-12
for _, row in H.iterrows():
    assert row["sum_dev"] <= tol
    assert row["min_w"] > 0
    assert row["split_tree"] <= tol
    assert row["split_pos"] <= tol
    assert row["diag_ivp_tree"] <= tol
    assert row["diag_ivp_pos"] <= tol
    assert row["scale_tree"] <= tol
    assert row["scale_pos"] <= tol
    assert row["perm_tree"] <= tol
    assert row["perm_valid"] == 208
    assert row["link_ok"] == 208
    assert row["qd_match"] == 208
    assert row["dist_ok"] == 208

print("Mechanism checks: PASSED at all 208 rebalances, both estimators, both bisections.")

ps = np.array(pos_spreads_208)
n_changes = sum(1 for i in range(1, len(root_sides)) if root_sides[i] != root_sides[i - 1])
print(f"positional permutation spread (K=20, sample Sigma): max={E3(ps.max())}  "
      f"n>1e-12={(ps > 1e-12).sum()}  n>1%={(ps > 0.01).sum()}")
print(f"median cophenetic correlation (sample Sigma, tree): {F4(float(np.median(cophs)))}")
print(f"root-split membership changes across 208 rebalances (HRP(S), tree): {n_changes}")

## §6 Hypothesis tests

### 6.1 — H1a / H1b (the UUP corner, from target weights)

In [ ]:
w_uup_hrp_ts = results["HRP(S)"]["wlog"]["UUP"]
w_uup_gmv_ts = results["GMV(S)"]["wlog"]["UUP"]
w_uup_mdp_ts = results["MDP(S)"]["wlog"]["UUP"]
w_uup_erc_ts = results["ERC(S)"]["wlog"]["UUP"]

lt_gmv = w_uup_hrp_ts < w_uup_gmv_ts
lt_mdp = w_uup_hrp_ts < w_uup_mdp_ts
h1a_full = lt_gmv & lt_mdp
n_full = len(h1a_full)
test_mask = h1a_full.index > split_ts
h1a_test = h1a_full.loc[test_mask]
n_test = int(test_mask.sum())
frac_full_h1a = float(h1a_full.mean())
frac_test_h1a = float(h1a_test.mean())

uup_2022 = {
    "HRP(S)": float(w_uup_hrp_ts.loc[asof_2022]),
    "GMV(S)": float(w_uup_gmv_ts.loc[asof_2022]),
    "MDP(S)": float(w_uup_mdp_ts.loc[asof_2022]),
    "ERC(S)": float(w_uup_erc_ts.loc[asof_2022]),
}
h1a_holds_2022 = uup_2022["HRP(S)"] < uup_2022["GMV(S)"] and uup_2022["HRP(S)"] < uup_2022["MDP(S)"]
h1a_verdict = ml.inference.count_rule(h1a_holds_2022, frac_full_h1a, frac_test_h1a)

print(f"H1a (HRP<GMV and HRP<MDP): full {int(h1a_full.sum())}/{n_full} ({100*frac_full_h1a:.1f}%), "
      f"test {int(h1a_test.sum())}/{n_test} ({100*frac_test_h1a:.1f}%)  -> {h1a_verdict}")
print(f"2022-12-31 w_UUP: HRP={uup_2022['HRP(S)']:.4f}  GMV={uup_2022['GMV(S)']:.4f}  "
      f"MDP={uup_2022['MDP(S)']:.4f}  ERC={uup_2022['ERC(S)']:.4f}")

h1b_full = w_uup_hrp_ts < w_uup_erc_ts
h1b_test = h1b_full.loc[test_mask]
print(f"\nH1b (descriptive, HRP<ERC): full {int(h1b_full.sum())}/{n_full} ({100*h1b_full.mean():.1f}%), "
      f"test {int(h1b_test.sum())}/{n_test} ({100*h1b_test.mean():.1f}%)")

### 6.2 — H2a / H2b (root singleton, correlation blindness)

Recomputed fresh (sample Σ) at every rebalance, including the Σ_cf
counterfactual — this is not read off the §4 backtests.

In [ ]:
h2_table = h.h2_counterfactual_table(panel, rdates)
print("h2_table computed:", h2_table.shape)

In [ ]:
n_h2a = int(h2_table["root_singleton"].sum())
n_both_singleton = int((h2_table["root_singleton"] & h2_table["root_singleton_cf"]).sum())
print(f"H2a (descriptive): UUP root singleton at {n_h2a}/208 rebalances")
print(f"both-singleton count (Sigma and Sigma_cf): {n_both_singleton}/208")

test_mask_h2 = h2_table.index > split_ts
h2b_verdicts = {}
for comp in ["GMV", "MDP", "ERC"]:
    holds = h2_table["dw_HRP"] < h2_table[f"dw_{comp}"]
    frac_full = float(holds.mean())
    frac_test = float(holds.loc[test_mask_h2].mean())
    holds_2022 = bool(holds.loc[asof_2022]) if asof_2022 in holds.index else bool(holds.iloc[-1])
    verdict = ml.inference.count_rule(holds_2022, frac_full, frac_test)
    h2b_verdicts[comp] = dict(frac_full=frac_full, frac_test=frac_test, holds_2022=holds_2022, verdict=verdict)
    print(f"H2b vs {comp}: full {int(holds.sum())}/208 ({100*frac_full:.1f}%), "
          f"test {int(holds.loc[test_mask_h2].sum())}/{int(test_mask_h2.sum())} ({100*frac_test:.1f}%), "
          f"2022-12-31 holds={holds_2022}  -> {verdict}")

### 6.3 — H3 (effective N, from target weights of the §4 backtests)

In [ ]:
effN_hrp_ts = 1.0 / (results["HRP(S)"]["wlog"] ** 2).sum(axis=1)
effN_gmv_ts = 1.0 / (results["GMV(S)"]["wlog"] ** 2).sum(axis=1)
effN_mdp_ts = 1.0 / (results["MDP(S)"]["wlog"] ** 2).sum(axis=1)
effN_erc_ts = 1.0 / (results["ERC(S)"]["wlog"] ** 2).sum(axis=1)

h3_specs = [
    ("GMV", effN_gmv_ts, lambda h, c: h > c),
    ("MDP", effN_mdp_ts, lambda h, c: h > c),
    ("ERC", effN_erc_ts, lambda h, c: h < c),
]
h3_verdicts = {}
for comp, comp_ts, op in h3_specs:
    holds = op(effN_hrp_ts, comp_ts)
    frac_full = float(holds.mean())
    frac_test = float(holds.loc[test_mask].mean())
    holds_2022 = bool(holds.loc[asof_2022])
    verdict = ml.inference.count_rule(holds_2022, frac_full, frac_test)
    h3_verdicts[comp] = dict(frac_full=frac_full, frac_test=frac_test, holds_2022=holds_2022, verdict=verdict)
    direction = ">" if op(1, 0) else "<"
    print(f"H3 effN(HRP) {direction} effN({comp}): full {int(holds.sum())}/208 ({100*frac_full:.1f}%), "
          f"test {int(holds.loc[test_mask].sum())}/{int(test_mask.sum())} ({100*frac_test:.1f}%), "
          f"2022-12-31 holds={holds_2022}  -> {verdict}")

print(f"\nmedian effN: HRP(S)={float(effN_hrp_ts.median()):.4f}  GMV(S)={float(effN_gmv_ts.median()):.4f}  "
      f"MDP(S)={float(effN_mdp_ts.median()):.4f}  ERC(S)={float(effN_erc_ts.median()):.4f}")
print(f"2022-12-31 effN: HRP={float(effN_hrp_ts.loc[asof_2022]):.4f}  GMV={float(effN_gmv_ts.loc[asof_2022]):.4f}  "
      f"MDP={float(effN_mdp_ts.loc[asof_2022]):.4f}  ERC={float(effN_erc_ts.loc[asof_2022]):.4f}")

### 6.4 — Paired daily-difference table (H4, H5, H6)

In [ ]:
pairs = [
    ("HRP(S)", "GMV(S)"),
    ("HRP(S)", "MDP(S)"),
    ("HRP(S)", "ERC(S)"),
    ("HRP(S)", "EW"),
    ("HRP(LW)", "HRP(S)"),
    ("HRP(S)", "IVP"),
    ("HRP[pos](S)", "HRP(S)"),
    ("HRP[ward](S)", "HRP(S)"),
]

paired_table, paired_series = ml.inference.paired_table(results, pairs, split_ts, summary_table)
print(paired_table.to_string(float_format=F4))

### 6.5 — CAPM (context only, not pre-registered) and the diff-regression Δα used for H4/H6

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_table = ml.inference.capm_table(results, list(strategies), MKT_simple, rf_daily)
print(capm_table.to_string(float_format=F4))

In [ ]:
diff_capm_table = ml.inference.diff_regression(paired_series, pairs, MKT_simple)
print(diff_capm_table.to_string(float_format=F4))

In [ ]:
decomp_results = ml.inference.beta_decomposition(capm_table, paired_table, diff_capm_table, pairs,
                                                 results, rf_daily, MKT_simple)
print(ml.inference.format_beta_decomposition(decomp_results, capm_table, diff_capm_table), end="")

### 6.6 — H7 (forecast bias: realized vs. ex-ante vol, HRP vs. GMV)

In [ ]:
h7_methods = ["HRP(S)", "GMV(S)", "MDP(S)", "ERC(S)", "IVP", "EW"]

q_table, n_dropped, sigma_cache = h.forecast_bias_table(panel, rdates, results, simple_returns, h7_methods)
print(f"segments dropped (< 10 remaining days after excluding cost-booking day): {n_dropped}")

In [ ]:
q_pivot = q_table.pivot(index="asof", columns="method", values="q")
test_mask_q = q_pivot.index > split_ts

print("mean q by method (full / test):")
q_means = pd.DataFrame({
    "full": q_pivot.mean(),
    "test": q_pivot.loc[test_mask_q].mean(),
})
print(q_means.to_string(float_format=F4))

d_h7 = (q_pivot["HRP(S)"] - q_pivot["GMV(S)"]).dropna()
d_full = d_h7
d_test = d_h7.loc[d_h7.index > split_ts]

t_full = float(d_full.mean() / (d_full.std(ddof=1) / np.sqrt(len(d_full))))
t_test = float(d_test.mean() / (d_test.std(ddof=1) / np.sqrt(len(d_test)))) if len(d_test) > 1 else float("nan")

if t_full <= -2 and float(d_test.mean()) < 0:
    h7_verdict = "SUPPORTED"
elif t_full <= -2:
    h7_verdict = "NOT ROBUST"
else:
    h7_verdict = "NOT SUPPORTED"

print(f"\nH7: d = q_HRP - q_GMV  full mean={float(d_full.mean()):.4f} t={t_full:.2f} n={len(d_full)}   "
      f"test mean={float(d_test.mean()):.4f} t={t_test:.2f} n={len(d_test)}  -> {h7_verdict}")

Reading note: every method's mean q is negative (realized vol below the 252-day ex-ante vol), so H7 is a RELATIVE statement — which method's forecast is least conservative — not evidence that any method's ex-ante vol is too low.

### 6.7 — VERDICT TABLE (mechanical application of the §2 rules)

In [ ]:
reg = ml.inference.load_registration("nb07")

d5 = decomp_results["HRP(S) - EW"]
t_full_ew = float(paired_table.loc[("HRP(S) - EW", "full"), "t"])
mean_test_ew = float(paired_table.loc[("HRP(S) - EW", "test (> split)"), "ann_mean_diff"])
t_full_lw = float(paired_table.loc[("HRP(LW) - HRP(S)", "full"), "t"])
t_test_lw = float(paired_table.loc[("HRP(LW) - HRP(S)", "test (> split)"), "t"])

stats = {
    "H1a": dict(holds_2022=h1a_holds_2022, frac_full=frac_full_h1a, frac_test=frac_test_h1a,
                display=f"full {int(h1a_full.sum())}/{n_full}, test {int(h1a_test.sum())}/{n_test}, "
                        f"2022-12-31 holds={h1a_holds_2022}"),
}
for comp in ["GMV", "MDP", "ERC"]:
    v = h2b_verdicts[comp]
    stats[f"H2b_{comp}"] = dict(holds_2022=v["holds_2022"], frac_full=v["frac_full"], frac_test=v["frac_test"],
                                display=f"full {100*v['frac_full']:.1f}%, test {100*v['frac_test']:.1f}%, 2022 holds={v['holds_2022']}")
for comp in ["GMV", "MDP", "ERC"]:
    v = h3_verdicts[comp]
    stats[f"H3_{comp}"] = dict(holds_2022=v["holds_2022"], frac_full=v["frac_full"], frac_test=v["frac_test"],
                               display=f"full {100*v['frac_full']:.1f}%, test {100*v['frac_test']:.1f}%, 2022 holds={v['holds_2022']}")
for hid, pair_key in [
    ("H4_GMV", "HRP(S) - GMV(S)"), ("H4_MDP", "HRP(S) - MDP(S)"), ("H4_ERC", "HRP(S) - ERC(S)"),
    ("H6_IVP", "HRP(S) - IVP"), ("H6_pos", "HRP[pos](S) - HRP(S)"), ("H6_ward", "HRP[ward](S) - HRP(S)"),
]:
    t = float(diff_capm_table.loc[pair_key, "t_alpha_diff"])
    stats[hid] = dict(t=t, display=f"t={t:.2f}")
stats["H5"] = dict(t_full=t_full_ew, mean_test=mean_test_ew, beta_part=d5["beta_part"], d_alpha=d5["d_alpha"],
                   display=f"t_full={t_full_ew:.2f}, mean_test={mean_test_ew:.4f}, "
                           f"|beta_part|={abs(d5['beta_part']):.4f} vs |d_alpha|={abs(d5['d_alpha']):.4f}")
stats["H6_LW"] = dict(t_full=t_full_lw, t_test=t_test_lw, display=f"t_full={t_full_lw:.2f}, t_test={t_test_lw:.2f}")
stats["H7"] = dict(t_full=t_full, mean_test=float(d_test.mean()),
                   display=f"t_full={t_full:.2f}, mean_test={float(d_test.mean()):.4f}")

verdict_table = ml.inference.evaluate(reg, stats)
print(verdict_table.to_string(index=False))

print("\n--- descriptive rows ---")
print(f"H1b (descriptive, HRP<ERC): full {int(h1b_full.sum())}/{n_full}, test {int(h1b_test.sum())}/{n_test}")
print(f"H2a (descriptive, UUP root singleton): {n_h2a}/208")
print(f"both-singleton count (Sigma and Sigma_cf): {n_both_singleton}/208")

## §7 Figures

### F3 — group-weight stackplot

Which asset groups carry the target weight over time under HRP(S) vs.
ERC(S) vs. MDP(S) vs. GMV(S)?

In [ ]:
fig = ml.plotting.group_stackplot(results, ["HRP(S)", "ERC(S)", "MDP(S)", "GMV(S)"], split_ts=split_ts)
plt.show()

hrp_group = ml.diagnostics.weights_by_group(results["HRP(S)"]["wlog"])
erc_group = ml.diagnostics.weights_by_group(results["ERC(S)"]["wlog"])
mdp_group = ml.diagnostics.weights_by_group(results["MDP(S)"]["wlog"])
gmv_group = ml.diagnostics.weights_by_group(results["GMV(S)"]["wlog"])

print("Mean target weight by asset group:")
group_means = pd.DataFrame({
    "HRP(S)": hrp_group.mean(), "ERC(S)": erc_group.mean(),
    "MDP(S)": mdp_group.mean(), "GMV(S)": gmv_group.mean(),
})
print(group_means.to_string(float_format=F4))

### F4 — effN, max weight, and UUP weight over time

Is the effN/UUP contrast between HRP, ERC, MDP and GMV a snapshot artifact
or does it hold across the full 208-rebalance history?

In [ ]:
fig = ml.plotting.concentration_panel(
    results, ["HRP(S)", "ERC(S)", "MDP(S)", "GMV(S)"], split_ts=split_ts,
    suptitle="HRP(S) vs ERC(S) vs MDP(S) vs GMV(S): effN, max weight, UUP weight",
)
plt.show()

### F5 — cumulative paired differences

For the Δα-registered pairs, is the beta-adjusted difference a persistent
drift or does it track the raw (total) difference? For HRP-EW and
HRP(LW)-HRP(S), is the total difference a persistent drift or a noisy path?

In [ ]:
alpha_pairs = [
    ("HRP(S)", "GMV(S)"), ("HRP(S)", "MDP(S)"), ("HRP(S)", "ERC(S)"),
    ("HRP(S)", "IVP"), ("HRP[pos](S)", "HRP(S)"), ("HRP[ward](S)", "HRP(S)"),
]
other_pairs = [("HRP(S)", "EW"), ("HRP(LW)", "HRP(S)")]

fig, axes = plt.subplots(2, 1, figsize=(9.5, 9), sharex=True)
ml.plotting.cumulative_paired(
    paired_series, alpha_pairs, split_ts,
    beta={f"{a} - {b}": diff_capm_table.loc[f"{a} - {b}", "beta_diff"] for a, b in alpha_pairs},
    mkt_excess=MKT_simple, ax=axes[0],
    title="(a) Delta-alpha registered pairs: solid = beta-adjusted, dashed = total",
)
ml.plotting.cumulative_paired(
    paired_series, other_pairs, split_ts, ax=axes[1],
    title="(b) HRP-EW and HRP(LW)-HRP(S): total", ylabel="cumulative diff (%)", legend_kw={"fontsize": 8},
)
plt.tight_layout(); plt.show()

### F6 — forecast bias over time

Is the HRP-vs-GMV ex-ante/realized vol gap concentrated in a particular
period, or roughly constant across the sample?

In [ ]:
fig = h.forecast_bias_figure(q_pivot, split_ts)
plt.show()

### F7 — root-split decomposition: why the corner comes back

Whenever UUP is a root singleton, HRP decides its weight in ONE comparison: σ²_UUP against V_rest, the variance of the inverse-variance portfolio of the other 12 assets (w_UUP = V_rest/(σ²_UUP &#43; V_rest)). How do those two quantities move over the 208 rebalances? Descriptive only — not a pre-registered test.

In [ ]:
f7_table = h.root_split_table(sigma_cache, h2_table, results, rdates)

root_mask = f7_table["root_singleton"]
max_dev = float((f7_table.loc[root_mask, "w_UUP_HRP"] - f7_table.loc[root_mask, "identity"]).abs().max())
assert max_dev <= 1e-12, f"root-singleton identity check failed: max_dev={max_dev:.3e}"
print(f"max |w_UUP(HRP) - identity| over root-singleton rebalances: {max_dev:.3e}")

fig = h.root_split_figure(f7_table, split_ts)
plt.show()

train_mask_f7 = f7_table.index <= split_ts
test_mask_f7 = f7_table.index > split_ts

f7_summary = pd.DataFrame({
    "train (<= split)": h.f7_summary_row(f7_table, train_mask_f7),
    "test (> split)": h.f7_summary_row(f7_table, test_mask_f7),
    "2022-12-31": h.f7_summary_row(f7_table, pd.Timestamp("2022-12-31")),
}).T
print(f7_summary.to_string(float_format=lambda v: f"{v:.4f}"))

## §8 TAKEAWAY

- **H1a — NOT ROBUST.** w_UUP(HRP) &lt; w_UUP(GMV) AND &lt; w_UUP(MDP) holds
  on 159/208 (76.4%) of all rebalances but only 14/40 (35.0%) of the test
  rebalances — it clears the &gt; 50%-of-208 bar but fails the
  &gt; 50%-of-test bar, so by the pre-registered count rule this is
  NOT ROBUST, not SUPPORTED. At 2022-12-31 the inequality does hold:
  w_UUP = HRP 0.5530, GMV 0.5812, MDP 0.5922, ERC 0.4836 — HRP's cash
  weight in UUP is smaller than GMV's and MDP's at that date, but the test
  window as a whole does not confirm the pattern out of sample.
- **H2b vs GMV — SUPPORTED.** &#124;w_UUP(Σ) − w_UUP(Σ_cf)&#124; for HRP is
  smaller than the same quantity for GMV on 100.0% of the full 208
  rebalances and 100.0% of the 40 test rebalances (2022-12-31 holds too).
- **H2b vs MDP — SUPPORTED.** Same comparison against MDP: 100.0% full,
  100.0% test.
- **H2b vs ERC — SUPPORTED.** Same comparison against ERC: 98.1% full,
  100.0% test. Next to these three counts: UUP is a root singleton under
  BOTH Σ and Σ_cf on 167/208 rebalances — whenever that holds, HRP's own
  &#124;Δw_UUP&#124; is exactly 0 by the root-singleton identity (see below),
  which is most of H2b's empirical content, not an estimated effect.
- **H3 vs GMV — SUPPORTED.** effN(HRP) &gt; effN(GMV) on 94.7% full, 90.0%
  test rebalances (2022-12-31 holds); median effN HRP(S)=5.03 vs
  GMV(S)=3.79.
- **H3 vs MDP — SUPPORTED.** effN(HRP) &gt; effN(MDP) on 100.0% full,
  100.0% test; median effN MDP(S)=3.35.
- **H3 vs ERC — SUPPORTED.** effN(HRP) &lt; effN(ERC) on 100.0% full,
  100.0% test; median effN ERC(S)=6.86 — HRP sits between GMV/MDP and ERC
  in concentration, as the pre-registered direction predicted.
- **H4 (HRP(S)−GMV(S)) — NULL HOLDS NARROWLY.** Δα from the direct
  diff-regression, t = 1.53.
- **H4 (HRP(S)−MDP(S)) — NULL HOLDS.** Δα t = 0.54.
- **H4 (HRP(S)−ERC(S)) — NULL HOLDS NARROWLY.** Δα t = 1.59.
- **H5 (HRP−EW, beta-driven) — NOT SUPPORTED.** The full-window paired t
  is −1.94, just short of the −2 threshold required by the pre-registered
  rule (the test-window mean is negative, −0.0357, and
  &#124;Δβ part&#124; = 0.0460 &gt; &#124;Δα&#124; = 0.0130, but the full-window t
  does not clear the bar, so H5 is not supported under the rule as
  written).
- **H6 (HRP(LW)−HRP(S), total) — NULL HOLDS.** t_full = 0.27, t_test =
  −0.31; larger-magnitude of the two is 0.31, well inside the null band —
  the Ledoit-Wolf sensitivity makes little difference to HRP's returns.
- **H6 (HRP(S)−IVP, Δα) — NULL HOLDS.** t = 1.01.
- **H6 (HRP&#91;pos&#93;(S)−HRP(S), Δα) — NULL HOLDS.** t = −0.33 — the bisection
  choice does not show up as a return difference despite the large weight
  differences documented in §3 and §5.
- **H6 (HRP&#91;ward&#93;(S)−HRP(S), Δα) — NULL HOLDS.** t = −0.51 — same
  conclusion for the linkage sensitivity selected by the pre-set recipe
  rule in §3.5.
- **H7 (forecast bias, HRP vs GMV) — SUPPORTED.** d = q_HRP − q_GMV has
  full-window t = −6.18 and a negative test-window mean (−0.0788) —
  every method's mean q is negative (full-window: ERC(S) −0.1049, EW
  −0.1885, GMV(S) −0.0356, HRP(S) −0.0999, IVP −0.1237, MDP(S) −0.0518),
  so the result is relative: GMV's ex-ante vol is the LEAST conservative
  forecast of the six, and HRP's is more conservative than GMV's by the
  printed mean d (−0.0643). The two optimized methods (GMV, MDP) have the
  least negative mean q, consistent with optimizer bias. The common
  negative shift across all methods is not explained by this and is not
  tested here.

**Descriptive (not pre-registered as directional tests):**
- **H1b (HRP vs ERC).** No direction was registered (the design-check
  snapshot was mixed). Descriptively, w_UUP(HRP) &lt; w_UUP(ERC) on 105/208
  (50.5%) full rebalances but only 3/40 (7.5%) test rebalances — essentially
  a coin flip full-sample and rare in the test window, consistent with no
  claim being registered here.
- **H2a.** UUP is a root singleton (alone on one side of the final merge)
  on 183/208 rebalances.

**Root-singleton mechanism.** At 2022-12-31 UUP's entire tree path is a
single split — (own-side n=1, other-side n=12, share to own side=0.553) —
so w_UUP(HRP) is exactly the root-singleton identity
V_rest/(σ²_UUP + V_rest), independent of UUP's correlations with the rest
of the universe; the printed check confirms
&#124;path product − w_UUP&#124; = 1.110e-16.

**Does HRP remove the corner, or just its correlation-driven part?** HRP
removes the identity's dependence on ρ for the specific asset that is a
root singleton — but the corner survives via the variance comparison
instead: at 2022-12-31, w_UUP is HRP 0.5530, GMV 0.5812, MDP 0.5922, ERC
0.4836 — all four methods still put more than 48% of the book in UUP.
HRP's cash weight is a bit smaller than GMV's and MDP's here (consistent
with H1a holding at this one date) but is not small in absolute terms, and
it is larger than ERC's.

**Concentration at that same date.** HRP's max RC share at 2022-12-31 is
0.5829 vs. ERC's 0.0769 (ERC's is 1/13 by construction, per notebook 06) —
HRP concentrates risk far more than ERC does even though both methods put
a similar fraction of NOTIONAL in UUP; HRP never targets equal risk shares
in the first place, so this is not a violated design goal, just a
difference between the two objectives.

**Root-split decomposition (F7, descriptive).** Over train rebalances (&lt;= split), median σ_UUP=0.0752, median √V_rest=0.0448, median ratio √V_rest/σ_UUP=0.5846, median ρ̄_rest=0.2409, median w_UUP(HRP)=0.2584, and UUP is a root singleton on 85.12% of train rebalances. Over test rebalances (&gt; split), median σ_UUP=0.0741, median √V_rest=0.0634, median ratio=1.0304, median ρ̄_rest=0.4440, median w_UUP(HRP)=0.5149, and the root-singleton share rises to 100.0%. At 2022-12-31: σ_UUP=0.0927, √V_rest=0.1031, ratio=1.1123, ρ̄_rest=0.4511, root singleton=1 — the maximum deviation between HRP's w_UUP and the root-singleton identity, over all root-singleton rebalances, is 2.220e-16. Between train and test the median ratio √V_rest/σ_UUP rises from 0.5846 to 1.0304 and the median ρ̄_rest from 0.2409 to 0.4440, alongside the rise in root-singleton frequency; at 2022-12-31 UUP's 55.30% weight is the single comparison of σ²_UUP against V_rest, printed above as the identity.

**CAPM alphas.** The single-strategy CAPM α/β table in §6.5 is, as in
notebooks 04–06, not pre-registered — shown for context only, with no
hypothesis attached to any individual number in it; it is Phase 2
multiple-testing material.

**No claim beyond what is tested here is made:** this notebook does not
test whether the HRP tree structure is "the right" clustering of this
universe, does not compare HRP against a risk-budgeted (non-equal)
tree-based alternative, and does not attribute UUP's persistent root
singleton to any specific macro regime.


## §9 Notes

- (a) HAP/HARP-style hierarchical allocators need an expected-return signal
  at each split, so they are return-based rather than risk-based and are
  not backtested in this notebook.
- (b) Hierarchical-clustering portfolios of the HCP type follow the tree
  down to a cluster cut and then allocate equal capital per cluster —
  HRP[tree] shares the tree-following step with this family, and the
  literature reports such portfolios as sensitive to the cluster-count
  choice.
- (c) The positional (equal-halves) bisection is the literature's original
  recursive-bisection algorithm, kept here only as a sensitivity — §1.2 and
  §5 show why tree-following is registered instead.